In [4]:
#task 5: Which issuing country has the highest number of transactions?

import pandas as pd
df = pd.read_csv("../../data/dabstep/data/context/payments.csv")

# step 1: load
print(df.shape)                                   
counts = df["issuing_country"].value_counts()

# step 2: count per country
print(counts)           

# step 3: answer, should be NL
print(counts.idxmax())                            

(138236, 21)
issuing_country
NL    29622
IT    28329
BE    23040
SE    21716
FR    14175
LU     7171
ES     7109
GR     7074
Name: count, dtype: int64
NL


In [6]:
#task 49: What is the top country (ip_country) for fraud? A. NL, B. BE, C. ES, D. FR

f = df[df["has_fraudulent_dispute"]]
by_count = f["ip_country"].value_counts()
by_volume = f.groupby("ip_country")["eur_amount"].sum()
by_rate = by_volume / df.groupby("ip_country")["eur_amount"].sum()
print(by_count, by_volume, by_rate, sep="\n\n")

ip_country
NL    2955
BE    2493
IT    1652
SE    1627
FR     843
LU     410
ES     407
GR     378
Name: count, dtype: int64

ip_country
BE    263833.85
ES     43531.87
FR     89135.03
GR     39916.73
IT    182231.72
LU     44628.44
NL    329134.08
SE    169937.79
Name: eur_amount, dtype: float64

ip_country
BE    0.122686
ES    0.067503
FR    0.068979
GR    0.062301
IT    0.070100
LU    0.067103
NL    0.121815
SE    0.084866
Name: eur_amount, dtype: float64


In [1]:
#task 1305: For account type H and the MCC description: Eating Places and Restaurants, what would be the average fee that the card scheme GlobalCard would charge for a transaction value of 10 EUR?

import json
import pandas as pd

BASE = "../../data/dabstep/data/context/"
mcc_df = pd.read_csv(BASE + "merchant_category_codes.csv")
with open(BASE + "fees.json") as f:
    fees = json.load(f)

# step 1: MCC lookup
hits = mcc_df[mcc_df["description"] == "Eating Places and Restaurants"]
assert len(hits) == 1, hits
mcc = int(hits["mcc"].iloc[0])
print("MCC:", mcc)

def applies(rule_list, value):
    return not rule_list or value in rule_list  # null / [] = wildcard

# step 2: matching rules
matching = [
    r for r in fees
    if r["card_scheme"] == "GlobalCard"
    and applies(r["account_type"], "H")
    and applies(r["merchant_category_code"], mcc)
]
print(len(matching), sorted(r["ID"] for r in matching))

# step 3: fee per rule for a 10 EUR transaction
value = 10
per_rule = {r["ID"]: r["fixed_amount"] + r["rate"] * value / 10000 for r in matching}
print(per_rule)

# step 4: unweighted average over matching rules
avg = sum(per_rule.values()) / len(per_rule)
print(f"{avg:.6f}")
assert f"{avg:.6f}" == "0.123217"

MCC: 5812
46 [5, 38, 92, 114, 140, 141, 160, 162, 192, 204, 221, 257, 267, 276, 280, 319, 325, 357, 359, 403, 427, 428, 456, 477, 498, 513, 556, 572, 612, 660, 666, 682, 688, 704, 709, 725, 741, 792, 813, 861, 888, 891, 892, 899, 917, 921]
{5: 0.199, 38: 0.139, 92: 0.112, 114: 0.175, 140: 0.228, 141: 0.098, 160: 0.15000000000000002, 162: 0.133, 192: 0.135, 204: 0.071, 221: 0.056, 257: 0.179, 267: 0.21500000000000002, 276: 0.11499999999999999, 280: 0.096, 319: 0.185, 325: 0.109, 357: 0.081, 359: 0.074, 403: 0.114, 427: 0.21600000000000003, 428: 0.05, 456: 0.07100000000000001, 477: 0.174, 498: 0.046, 513: 0.15200000000000002, 556: 0.177, 572: 0.104, 612: 0.089, 660: 0.107, 666: 0.15500000000000003, 682: 0.114, 688: 0.179, 704: 0.103, 709: 0.137, 725: 0.019, 741: 0.078, 792: 0.10300000000000001, 813: 0.14600000000000002, 861: 0.067, 888: 0.102, 891: 0.189, 892: 0.116, 899: 0.16999999999999998, 917: 0.108, 921: 0.032}
0.123217


In [2]:
#task 1753: What are the applicable fee IDs for Belles_cookbook_store in March 2023?

import json
import pandas as pd

BASE = "../../data/dabstep/data/context/"
MERCHANT = "Belles_cookbook_store"

# step 1: load data
df = pd.read_csv(BASE + "payments.csv")
with open(BASE + "merchant_data.json") as f:
    merchant_data = json.load(f)
with open(BASE + "fees.json") as f:
    fees = json.load(f)

merchant = next(m for m in merchant_data if m["merchant"] == MERCHANT)
print(merchant)  # check the key names here if a KeyError appears below

# step 2: March 2023 transactions (2023 is not a leap year: March = day_of_year 60-90)
march = df[
    (df["merchant"] == MERCHANT)
    & (df["year"] == 2023)
    & (df["day_of_year"].between(60, 90))
].copy()
march["intracountry"] = march["issuing_country"] == march["acquirer_country"]
print("March transactions:", len(march))

# step 3: merchant-level attributes
account_type = merchant["account_type"]
capture_delay = str(merchant["capture_delay"])
mcc = int(merchant["merchant_category_code"])
print(account_type, capture_delay, mcc)

# step 4: monthly volume (EUR) and fraud ratio (fraud volume / total volume, in %)
monthly_volume = march["eur_amount"].sum()
fraud_volume = march.loc[march["has_fraudulent_dispute"], "eur_amount"].sum()
fraud_level = 100 * fraud_volume / monthly_volume
print(f"volume: {monthly_volume:.2f} EUR, fraud level: {fraud_level:.4f}%")

# helpers for rule matching
def applies(rule_list, value):
    return not rule_list or value in rule_list  # null / [] = wildcard

def parse_num(s):
    s = s.strip().lower().rstrip("%")
    mult = 1
    if s.endswith("k"):
        mult, s = 1e3, s[:-1]
    elif s.endswith("m"):
        mult, s = 1e6, s[:-1]
    return float(s) * mult

def in_range(rule_str, value):
    if rule_str is None:
        return True
    s = rule_str.strip()
    if s.startswith("<"):
        return value < parse_num(s[1:])
    if s.startswith(">"):
        return value > parse_num(s[1:])
    lo, hi = s.split("-")
    return parse_num(lo) <= value <= parse_num(hi)  # boundary inclusivity is an assumption

def capture_matches(rule_val, merchant_val):
    if rule_val is None:
        return True
    if merchant_val.isdigit():
        d = int(merchant_val)
        bucket = "<3" if d < 3 else "3-5" if d <= 5 else ">5"
        return rule_val == bucket
    return rule_val == merchant_val  # 'immediate' / 'manual'

# step 5: rules matching the merchant-level conditions
merchant_rules = [
    r for r in fees
    if applies(r["account_type"], account_type)
    and applies(r["merchant_category_code"], mcc)
    and capture_matches(r["capture_delay"], capture_delay)
    and in_range(r["monthly_volume"], monthly_volume)
    and in_range(r["monthly_fraud_level"], fraud_level)
]
print(len(merchant_rules), sorted(r["ID"] for r in merchant_rules))

# step 6: keep rules that match at least one March transaction
def txn_mask(rule, txns):
    mask = pd.Series(True, index=txns.index)
    if rule["card_scheme"]:
        mask &= txns["card_scheme"] == rule["card_scheme"]
    if rule["is_credit"] is not None:
        mask &= txns["is_credit"] == rule["is_credit"]
    if rule["aci"]:
        mask &= txns["aci"].isin(rule["aci"])
    if rule["intracountry"] is not None:
        mask &= txns["intracountry"] == bool(rule["intracountry"])
    return mask

applicable = [r["ID"] for r in merchant_rules if txn_mask(r, march).any()]
print(len(applicable), sorted(applicable))

# step 7: compare with the public answer (set comparison, order does not matter)
expected = {384, 394, 276, 150, 536, 286, 163, 36, 680, 939, 428, 813, 556, 51, 53,
            572, 960, 64, 709, 454, 595, 725, 473, 347, 477, 608, 868, 741, 231, 107,
            626, 249, 123, 381}
print("missing:", sorted(expected - set(applicable)))
print("extra:  ", sorted(set(applicable) - expected))
assert set(applicable) == expected

{'merchant': 'Belles_cookbook_store', 'capture_delay': '1', 'acquirer': ['lehman_brothers'], 'merchant_category_code': 5942, 'account_type': 'R'}
March transactions: 1277
R 1 5942
volume: 116436.32 EUR, fraud level: 10.2488%
47 [36, 51, 53, 64, 65, 80, 107, 123, 150, 163, 183, 231, 249, 276, 286, 304, 347, 381, 384, 394, 428, 454, 473, 477, 498, 536, 556, 572, 595, 608, 626, 631, 678, 680, 709, 725, 741, 813, 849, 861, 868, 871, 892, 924, 939, 942, 960]
34 [36, 51, 53, 64, 107, 123, 150, 163, 231, 249, 276, 286, 347, 381, 384, 394, 428, 454, 473, 477, 536, 556, 572, 595, 608, 626, 680, 709, 725, 741, 813, 868, 939, 960]
missing: []
extra:   []
